# 03. Determine DECam Streak's Exact Location
Written by Kiyoaki Okudaira and Meredith Rawls<br>
*University of Washington / IAU CPS SatHub<br>
(kiyoaki@uw.edu or okudaira.kiyoaki.528@s.kyushu-u.ac.jp)<br>
<br>
This code is written for ASTR 499 undergraduate research with Dr. Meredith.<br>
Determine DECam Streak's exact location (RA/DEC and x/y coordinates)<br>
<br>
**History**<br>
coding 2026-02-21 : 1st coding<br>
update 2026-05-02 : output filename changed & parallel processing

### Import and initial settings
**Standard libraries**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from os import path
import pickle

from scipy.stats import sigmaclip

from astropy.io import fits
from astropy.wcs import WCS

from tqdm.notebook import tqdm

**Satphotometry library**

In [ ]:
import satphotometry as sp

**Import file settings**

In [ ]:
# project directory
PATH_project = '/astro/store/shire/kiyoaki/ASTR499/'

PATH_input   = PATH_project + 'input/'
PATH_output  = PATH_project + 'output/'

PATH_image   = PATH_input  + 'fits_data/'
PATH_session = PATH_output + 'session/'

# ORIGINAL streak list from Alex
fname_streak_list = 'streaks_augmented_20230817.csv'
PATH_streak_list  = fname_streak_list + 'decam_streak_list/' + fname_streak_list

**Process method setting**

In [ ]:
paralell_process = True
if paralell_process:
    from concurrent.futures import ThreadPoolExecutor, as_completed

### Load Streak Dataset
Dataset is grouped by EXPNUM because if the EXPNUM is the same, the request to satchecker is the same．

In [ ]:
with open(PATH_session+path.splitext(path.basename(fname_streak_list))[0]+'_02_masked_by_gauss.pkl', 'rb') as f:
    streak_table = pickle.load(f)
streak_table = streak_table.group_by("expnum")

## Streak rotation and position
**Canvas**

In [ ]:
sc_ra = [None] * len(streak_table)
ec_ra = [None] * len(streak_table)
s1_ra = [None] * len(streak_table)
s2_ra = [None] * len(streak_table)
e1_ra = [None] * len(streak_table)
e2_ra = [None] * len(streak_table)

sc_dec = [None] * len(streak_table)
ec_dec = [None] * len(streak_table)
s1_dec = [None] * len(streak_table)
s2_dec = [None] * len(streak_table)
e1_dec = [None] * len(streak_table)
e2_dec = [None] * len(streak_table)

streak_y_list     = [None] * len(streak_table)
streak_x_min_list = [None] * len(streak_table)
streak_x_max_list = [None] * len(streak_table)
sigma_mean_list   = [None] * len(streak_table)

**Streak position**<br>
Determine accurate streak location (RA,DEC & x,y)

In [ ]:
for row in tqdm(streak_table):
    try:
        streak_ID = row["streakID"]

        basename = path.basename(row["archive_filename"])
        md5sum = row["md5sum"]
        expnum = row["expnum"]
        ccdnum = row["ccdnum"]

        idx = np.where(streak_table["streakID"] == streak_ID)[0][0]

        save_path = PATH_image + path.splitext(path.splitext(basename)[0])[0] + "_CCD_{0}".format(ccdnum) + path.splitext(path.splitext(basename)[0])[1] + path.splitext(basename)[1]

        section   = row["gauss_y"][:-2]
        amplitude = row["amplitude"][:-2]
        ground    = row["ground"][:-2]
        mu        = row["mu"][:-2]
        sigma     = row["sigma"][:-2]
        s_n       = (amplitude+ground)/ground

        mu_initial_mask = (mu > 50) & (mu < 150) & (sigma > 0)

        ground_mask = mu_initial_mask
        for i in range(0,1):
            _,low,high = sigmaclip(ground[ground_mask],3,3)
            ground_mask = (ground > low) & (ground < high)

        s_n_mask = mu_initial_mask & (s_n > 1)
        for i in range(0,1):
            _,low,high = sigmaclip(s_n[s_n_mask],3,3)
            s_n_mask = (s_n > low) & (s_n < high) & (s_n > 1)

        for i in range(0,1):
            _,low,high = sigmaclip(mu[mu_initial_mask],2,2)
            mu_mask = (mu > low) & (mu < high)

        sigma_mask = mu_initial_mask
        for i in range(0,1):
            _,low,high = sigmaclip(sigma[mu_initial_mask],3,3)
            sigma_mask = (sigma > low) & (sigma < high)

        mask = ground_mask & s_n_mask & mu_mask & sigma_mask & mu_initial_mask

        streak_x_min = section[mask][0]
        streak_x_max = section[mask][-1] + 49
        streak_y     = np.mean(mu[mask])
        sigma_mean   = np.mean(sigma[mask])

        s_n_mean = np.mean(s_n[s_n_mask])

        # Read Image file
        hdu = fits.open(save_path)
        main_header = hdu[0].header
        ccd_header = hdu[1].header
        ccd_data = hdu[1].data
        ccd_wcs = WCS(ccd_header)

        ccd_data = fits.getdata(save_path,1)
        rotated_img,r_matrix,r_pararell = sp.imgrotation.rotate_image(ccd_data, row["rt_angle"], row["initial_streak_coord"], True, 100, True)

        # # plot (debug purpose)
        # y_pix = section + 25

        # fig, ax = plt.subplots(
        #     4, 1,
        #     figsize=(8, 9.6),
        #     sharex=True,
        #     gridspec_kw={"hspace": 0.05}
        # )

        # # amplitude & background
        # # ax[0].scatter(y_pix, amplitude+ground, s=12, label="amplitude+bkg")
        # # ax[0].scatter(y_pix, ground, s=12, label="bkg")
        # ax[0].scatter(y_pix[mask], (amplitude+ground)[mask], s=8, marker='+', label="amplitude+bkg masked")
        # ax[0].scatter(y_pix[mask], ground[mask], s=8, marker='+', label="bkg masked")
        # ax[0].set_ylabel("count [HDU]")
        # ax[0].legend()
        # ax[0].grid()

        # # S/N
        # # ax[1].scatter(y_pix, s_n, s=12)
        # ax[1].scatter(y_pix[mask], s_n[mask], s=8, marker='+', color="red", label="masked")
        # ax[1].set_ylabel("s/n ratio")
        # ax[1].grid()
        # ax[1].legend()

        # # mu ± 3σ
        # ax[2].errorbar(
        #     y_pix[sigma>0],
        #     mu[sigma>0],
        #     yerr=3*np.array(sigma[sigma>0]),
        #     fmt='o',
        #     markersize=4,
        #     capsize=3,
        #     elinewidth=1,
        #     label=r"$\mu \pm 3\sigma$"
        # )
        # ax[2].scatter(y_pix[mask], mu[mask], s=20, marker='x', color="red", label="masked")

        # ax[2].set_xlabel("y [pix]")
        # ax[2].set_ylabel("mu [pix]")
        # ax[2].legend()
        # ax[2].grid()

        # ax[3].imshow(rotated_img, cmap='gray', origin='lower', vmin=np.percentile(rotated_img, 5), vmax=np.percentile(rotated_img, 95))

        # fig.suptitle("Exposure No.{0} / CCD No.{1} | Streak No.{2}".format(expnum,ccdnum,streak_ID))
        # plt.show()

        # rotation
        def _inv_rot(x, y, r_matrix, r_pararell, wcs):
            p = np.array([x, y], dtype=float)
            src_xy = r_matrix @ (p - r_pararell)
            src_radec = wcs.pixel_to_world(src_xy[0], src_xy[1])
            return src_xy,src_radec
        
        # streak location
        streak_sc_src_xy,streak_sc_radec = _inv_rot(streak_x_min, streak_y, r_matrix, r_pararell, ccd_wcs)
        streak_ec_src_xy,streak_ec_radec = _inv_rot(streak_x_max, streak_y, r_matrix, r_pararell, ccd_wcs)

        streak_s1_src_xy,streak_s1_radec = _inv_rot(streak_x_min, streak_y-3*sigma_mean, r_matrix, r_pararell, ccd_wcs)
        streak_s2_src_xy,streak_s2_radec = _inv_rot(streak_x_min, streak_y+3*sigma_mean, r_matrix, r_pararell, ccd_wcs)
        streak_e1_src_xy,streak_e1_radec = _inv_rot(streak_x_max, streak_y-3*sigma_mean, r_matrix, r_pararell, ccd_wcs)
        streak_e2_src_xy,streak_e2_radec = _inv_rot(streak_x_max, streak_y+3*sigma_mean, r_matrix, r_pararell, ccd_wcs)

        sc_ra[idx] = streak_sc_radec.ra.deg
        ec_ra[idx] = streak_ec_radec.ra.deg
        s1_ra[idx] = streak_s1_radec.ra.deg
        s2_ra[idx] = streak_s2_radec.ra.deg
        e1_ra[idx] = streak_e1_radec.ra.deg
        e2_ra[idx] = streak_e2_radec.ra.deg

        sc_dec[idx] = streak_sc_radec.dec.deg
        ec_dec[idx] = streak_ec_radec.dec.deg
        s1_dec[idx] = streak_s1_radec.dec.deg
        s2_dec[idx] = streak_s2_radec.dec.deg
        e1_dec[idx] = streak_e1_radec.dec.deg
        e2_dec[idx] = streak_e2_radec.dec.deg

        streak_y_list[idx]     = streak_y
        streak_x_min_list[idx] = streak_x_min
        streak_x_max_list[idx] = streak_x_max
        sigma_mean_list[idx]   = sigma_mean
    except:
        pass

# write to canvas
streak_table["sc_ra"] = sc_ra
streak_table["ec_ra"] = ec_ra
streak_table["s1_ra"] = s1_ra
streak_table["s2_ra"] = s2_ra
streak_table["e1_ra"] = e1_ra
streak_table["e2_ra"] = e2_ra

streak_table["sc_dec"] = sc_dec
streak_table["ec_dec"] = ec_dec
streak_table["s1_dec"] = s1_dec
streak_table["s2_dec"] = s2_dec
streak_table["e1_dec"] = e1_dec
streak_table["e2_dec"] = e2_dec

streak_table["streak_y"]     = streak_y_list
streak_table["streak_x_min"] = streak_x_min_list
streak_table["streak_x_max"] = streak_x_max_list
streak_table["streak_sigma"] = sigma_mean_list

# save
with open(PATH_session+path.splitext(path.basename(fname_streak_list))[0]+'_02_masked_by_gauss.pkl', 'wb') as f:
    pickle.dump(streak_table, f)